In [1]:
import json
import pandas as pd

base_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI"
json_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-001\eeg\sub-001_task-fmrieoec_eeg.json"
tsv_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-001\eeg\sub-001_task-fmrieoec_channels.tsv"

with open(json_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Metadatos JSON:")
for k, v in meta.items():
    print(f"{k}: {v}")

Metadatos JSON:
ECGChannelCount: 1
EEGChannelCount: 32
EEGReference: Cz
EMGChannelCount: 0
EOGChannelCount: 0
EpochLength: 444.594
InstitutionAddress: Blvd. Juriquilla 3001, Juriquilla, Santiago de Queretaro, Queretaro, 76230, Mexico
InstitutionName: Universidad Nacional Autonoma de Mexico
InstitutionalDepartmentName: Instituto de Neurobiologia
MiscChannelCount: 0
PowerLineFrequency: 60
RecordingDuration: 444.594
SamplingFrequency: 1000
SoftwareFilters: n/a
TaskDescription: Eyes closed and eyes open EEG data recorded outside the MR environment, inside the scanner without image acquisition and during simultaneous fMRI acquisition
TaskName: fmrieoec
TriggerChannelCount: 0


In [ ]:
channels = pd.read_csv(tsv_path, sep="\t")
print("\nCanales:")
print(channels.head())
print("\nTipos de canal:")
print(channels["type"].value_counts())

In [ ]:
import mne

set_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-001\eeg\sub-001_task-fmrieoec_eeg.set"
raw = mne.io.read_raw_eeglab(set_path, preload=True)

print(raw)
print(raw.info)
print(raw.ch_names)

In [ ]:
mne.viz.set_browser_backend("qt")

raw.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True
)

In [ ]:
from mne.preprocessing import ICA

raw.filter(l_freq=1.0, h_freq=40.0)
raw.notch_filter(freqs=50)

ica = ICA(n_components=20, random_state=97, max_iter="auto")
ica.fit(raw)

In [ ]:
ica.plot_sources(raw)
ica.plot_components()

In [ ]:
%matplotlib qt

In [ ]:
import nibabel as nib
from nibabel.viewers import OrthoSlicer3D
from nilearn import image

fmri_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_MRI\sub-001\ses-001\func\sub-001_ses-001_task-eoec_bold.nii.gz"
anat_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_MRI\sub-001\ses-001\anat\sub-001_ses-001_acq-highres_T1w.nii.gz"

fmri_img = nib.load(fmri_path)
anat_img = nib.load(anat_path)

# Anatomica en ventana aparte
anat_viewer = OrthoSlicer3D(anat_img.get_fdata(), anat_img.affine)
anat_viewer.show()

# Volumen funcional t=0 en ventana aparte
vol0 = image.index_img(fmri_img, 0)
func_viewer = OrthoSlicer3D(vol0.get_fdata(), vol0.affine)
func_viewer.show()

In [ ]:
from interactive_fmri_viewer import launch_interactive_viewer

fmri_path = r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_MRI\sub-001\ses-001\func\sub-001_ses-001_task-eoec_bold.nii.gz"
launch_interactive_viewer(fmri_path)

In [ ]:
def describe_image(path: Path) -> nib.spatialimages.SpatialImage:
    img = nib.load(str(path))
    data = img.get_fdata(dtype=np.float32)
    header = img.header

    print(f"\n=== {path.name} ===")
    print(f"Ruta: {path}")
    print(f"Shape: {img.shape}")
    print(f"Affine:\n{img.affine}")
    print(f"Voxel size (mm): {header.get_zooms()}")
    print(f"Datatype: {header.get_data_dtype()}")
    print(f"Min/Max: {np.nanmin(data):.3f} / {np.nanmax(data):.3f}")

    if data.ndim == 4:
        mean_img = data.mean(axis=3)
        print(f"TR / pixdim[4]: {header.get_zooms()[3] if len(header.get_zooms()) > 3 else 'N/A'}")
        print(f"Volumenes: {data.shape[3]}")
        print(f"Media global: {np.nanmean(mean_img):.3f}")
        print(f"Std global: {np.nanstd(mean_img):.3f}")
    else:
        print(f"Media global: {np.nanmean(data):.3f}")
        print(f"Std global: {np.nanstd(data):.3f}")

    return img

In [ ]:
from pathlib import Path

fmri_path_describe = Path(fmri_path)
describe_image(fmri_path_describe)

In [ ]:
anat_path_describe = Path(anat_path)
describe_image(anat_path_describe)